# Wprowadzenie do NLP: Jak działa „polka” i modele językowe?

**Cel zajęć:** Dziś zajrzymy pod maskę sztucznej inteligencji. Dowiecie się, dlaczego ChatGPT czasem "zmyśla", jak komputer rozumie memy i jak możemy sterować AI za pomocą odpowiednich poleceń (promptów).

**💡 Główna idea wykładu:**
> Model językowy to nie jest wszechwiedząca wyrocznia. To potężna, zasilana matematyką "klawiatura w telefonie", która po prostu stara się przewidzieć **następne słowo**.

Będziemy pracować na polskim modelu **eryk-mazus/polka-1.1b**.

## 0. Instalacja i importy

Pobieramy model z Hugging Face: `eryk-mazus/polka-1.1b`.


In [3]:
import torch
import torch.nn.functional as F
from transformers import pipeline, set_seed
from transformers import logging as transformers_logging

transformers_logging.set_verbosity_error()


generator = pipeline("text-generation", model="eryk-mazus/polka-1.1b")

tokenizer = generator.tokenizer
model = generator.model
model.eval()

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

set_seed(42)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

## Część 1. Jak komputer widzi tekst? (Tokenizacja)

My widzimy zdania i słowa. Komputer widzi tylko liczby. Zanim tekst trafi do modelu, jest szatkowany na mniejsze kawałki zwane **tokenami**. Token to nie zawsze całe słowo! Często to sylaba, przedrostek, a nawet sama spacja.

Zaraz zobaczycie, jak nasz model "kroi" polskie zdania.

In [4]:
def pokaz_tokeny(tekst):
    ids = tokenizer.encode(tekst)

    print("TEKST:")
    print(tekst)

    print("TOKENY I ICH ID:")
    for i, idx in enumerate(ids, start=1):
        token_surowy = tokenizer.convert_ids_to_tokens([idx])[0]
        token_czytelny = tokenizer.decode([idx])
        print(f"{i:02d}. {repr(token_czytelny):20s} -> id: {idx:5d}   surowo: {repr(token_surowy)}")

    print("Liczba tokenów:", len(ids))

tekst = "Lubię matematykę, ale nie lubię sprawdzianów."
pokaz_tokeny(tekst)


TEKST:
Lubię matematykę, ale nie lubię sprawdzianów.
TOKENY I ICH ID:
01. '<s>'                -> id:     1   surowo: '<s>'
02. 'Lub'                -> id: 19278   surowo: '▁Lub'
03. 'ię'                 -> id:  9497   surowo: 'ię'
04. 'mat'                -> id:  1775   surowo: '▁mat'
05. 'emat'               -> id:  4579   surowo: 'emat'
06. 'yk'                 -> id: 12072   surowo: 'yk'
07. 'ę'                  -> id: 30023   surowo: 'ę'
08. ', ale'              -> id: 40441   surowo: ',▁ale'
09. 'nie'                -> id:  4930   surowo: '▁nie'
10. 'lub'                -> id: 14757   surowo: '▁lub'
11. 'ię'                 -> id:  9497   surowo: 'ię'
12. 'sprawdz'            -> id: 40364   surowo: '▁sprawdz'
13. 'ian'                -> id:   713   surowo: 'ian'
14. 'ów'                 -> id:  2165   surowo: 'ów'
15. '.'                  -> id: 29889   surowo: '.'
Liczba tokenów: 15


## Część 2. AI jako "przewidywacz" słów

Model dostaje początek zdania (Prompt) i zgaduje, co powinno być dalej, licząc prawdopodobieństwo. Nie ma gotowej bazy odpowiedzi!


In [5]:
def top_nastepne_tokeny(prompt, top_k=10):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits[0, -1, :]
        probs = F.softmax(logits, dim=-1)
        values, indices = torch.topk(probs, k=top_k)

    print("PROMPT:")
    print(repr(prompt))
    print("Najbardziej prawdopodobne następne tokeny:")
    for prob, idx in zip(values, indices):
        token = tokenizer.decode([idx.item()])
        print(f"{repr(token):18s}  prawdopodobieństwo ~ {prob.item():.4f}")

top_nastepne_tokeny("Stolicą Polski jest", top_k=10)


PROMPT:
'Stolicą Polski jest'
Najbardziej prawdopodobne następne tokeny:
'Warszawa'          prawdopodobieństwo ~ 0.5078
'...'               prawdopodobieństwo ~ 0.0664
'W'                 prawdopodobieństwo ~ 0.0645
'Krak'              prawdopodobieństwo ~ 0.0605
'st'                prawdopodobieństwo ~ 0.0223
'…'                 prawdopodobieństwo ~ 0.0210
'R'                 prawdopodobieństwo ~ 0.0127
'Kat'               prawdopodobieństwo ~ 0.0123
'Warsza'            prawdopodobieństwo ~ 0.0112
'-'                 prawdopodobieństwo ~ 0.0106


## Część 3. Możliwe parametry do ustawienia

Teraz zobaczymy pełniejsze generowanie oraz jakie parametry możemy ustawiać! Model będzie dopisywał tekst do promptu.

Najważniejsze parametry:

- `max_new_tokens` — ile nowych tokenów model ma dopisać,
- `temperature` — jak bardzo odpowiedzi mają być losowe/kreatywne,
- `do_sample=False/True` — model wybiera najbardziej prawdopodobny token/losuje spośród tokenów

In [6]:
def generuj(prompt, max_new_tokens=40, temperature=0.8, do_sample=True, top_k=50, top_p=0.95, num_return_sequences=1):
    outputs = generator(
        prompt,
        max_new_tokens=max_new_tokens,
        max_length=None,
        do_sample=do_sample,
        temperature=temperature,
        top_k=top_k,
        top_p=top_p,
        num_return_sequences=num_return_sequences,
        pad_token_id=tokenizer.eos_token_id,
    )

    if num_return_sequences == 1:
        return outputs[0]["generated_text"]
    return [x["generated_text"] for x in outputs]

prompt = "Dzisiaj na lekcji informatyki dowiedziałem się, że"
print(generuj(prompt, max_new_tokens=50, temperature=0.8, do_sample=True))


Dzisiaj na lekcji informatyki dowiedziałem się, że baterie w smartfonie mają taki skład, że naładowanie ich do pełna zajmuje im tylko 10 minut. To mnie bardzo zdziwiło, ponieważ zazwyczaj naładow


In [7]:
prompt = "Na wakacje chciałbym pojechać do"

print("Niska temperatura:")
print(generuj(prompt, max_new_tokens=35, temperature=0.2, do_sample=True))

print("\nŚrednia temperatura:")
print(generuj(prompt, max_new_tokens=35, temperature=0.8, do_sample=True))

print("\nWysoka temperatura:")
print(generuj(prompt, max_new_tokens=35, temperature=1.5, do_sample=True))

Niska temperatura:
Na wakacje chciałbym pojechać do Hiszpanii. Co warto zobaczyć? | Blog podróżniczy\nHome » Podróże » Na wakacje chciałbym pojecha

Średnia temperatura:
Na wakacje chciałbym pojechać do Czech i Niemiec. Z jakiego miasta warto wiedzieć, że mieszkają tam bardzo ciekawi ludzie. Co mógłbym wzią

Wysoka temperatura:
Na wakacje chciałbym pojechać do Grecji, chciałabym przy okazji odkryć to fantastyczne zamek Grecja - Forum podróżników - jacybylejs


## Generowanie ręczne: patrzymy na każdy krok

Ta funkcja pokazuje, że tekst powstaje stopniowo. W każdej iteracji model wybiera jeden następny token.

In [ ]:
def generuj_krok_po_kroku(prompt, kroki=12, temperature=0.8):
    text = prompt
    print("START:", repr(text))
    print("-" * 60)

    for step in range(1, kroki + 1):
        inputs = tokenizer(text, return_tensors="pt").to(model.device)
        with torch.no_grad():
            logits = model(**inputs).logits[0, -1, :] / temperature
            probs = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1).item()

        next_token = tokenizer.decode([next_id])
        text += next_token
        print(f"Krok {step:02d}: dodano {repr(next_token)}")
        print(text)
        print("-" * 60)

    return text

_ = generuj_krok_po_kroku("Sztuczna inteligencja jest", kroki=10, temperature=0.8)


START: 'Sztuczna inteligencja jest'
------------------------------------------------------------
Krok 01: dodano 'bez'
Sztuczna inteligencja jestbez
------------------------------------------------------------


## Część 4. Prompt Engineering (Few-Shot)

Komputer nie ma magicznego przycisku "rozwiąż zadanie". Musimy nadać mu kontekst. Małe modele (jak nasza Papuga) najlepiej uczą się przez naśladownictwo.

### Nowe pojęcia!

- **Zero-shot** — prosimy o zadanie bez przykładów.
- **One-shot** — dajemy jeden przykład.
- **Few-shot** — dajemy kilka przykładów.


In [ ]:
def pokaz(
    prompt,
    max_new_tokens=50,
    temperature=0.8,
    top_k=50,
    top_p=0.95,
    do_sample=True,
):
    wynik = generator(
        prompt,
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        temperature=temperature,
        top_k=top_k,
        top_p=top_p,
        pad_token_id=tokenizer.eos_token_id,
        return_full_text=True
    )[0]["generated_text"]

    dopisek = wynik[len(prompt):]

    return wynik, dopisek

In [ ]:
zero_shot = """Tekst: "Ten przepis jest świetny!"
Sentyment:"""

wynik, dopisek = pokaz(zero_shot, max_new_tokens=50, temperature=0.1)

print(wynik)

In [ ]:
few_shot = """Tekst: "Nienawidzę smerfów!"
Sentyment: Negatywny
###
Tekst: "Jaki piękny dzień"
Sentyment: Pozytywny
###
Tekst: "Jutro idę do kina"
Sentyment: Neutralny
###
Tekst: "Ten przepis jest świetny!"
Sentyment:"""

wynik, dopisek = pokaz(few_shot, max_new_tokens=50, temperature=0.1)

print(wynik)
print(dopisek.split()[0])
